# Customer Support Triage Agent — demo end-to-end

Runs the full LangGraph flow on a few tickets. Without `ANTHROPIC_API_KEY` and the `ml` extra every component uses its **deterministic fallback** (keyword classifier, heuristic priority, template reply, hashed encoder); the outputs below are labelled accordingly and are not the production system's quality.

In [1]:
import asyncio, json, os
from pathlib import Path
os.environ.pop("ANTHROPIC_API_KEY", None)          # offline: deterministic fallback
os.environ["USE_LOCAL_CLASSIFIER"] = "false"
from src.config import Settings
from src.api.schemas import TicketIn
from src.eval.runner import build_offline_pipeline, run_pipeline_eval, load_cases, format_report
settings = Settings(anthropic_api_key=None, use_local_classifier=False)
pipeline = asyncio.run(build_offline_pipeline(settings))
print("classifier:", pipeline.classifier.name, "| llm_enabled:", pipeline.llm_enabled)

classifier: keyword_heuristic | llm_enabled: False


In [2]:
tickets = [
    ("t-1", "Where is my order #66721? It has been a week."),
    ("t-2", "This is unacceptable, I want to file a complaint about the courier."),
    ("t-3", "What is your refund policy on opened items?"),
]
for tid, body in tickets:
    out = asyncio.run(pipeline.triage(TicketIn(ticket_id=tid, body=body)))
    print(
        f"{tid}: intent={out.intent} ({out.intent_confidence:.2f}) priority={out.priority} "
        f"sentiment={out.sentiment} -> {out.decision.decision} "
        f"conf={out.decision.confidence:.2f}"
    )
    print("   draft:", out.draft_response[:110], "...")
    print("   trace:", out.trace_id, "| tokens:", out.input_tokens, out.output_tokens,
          "| cost:", out.cost_usd)

t-1: intent=track_order (0.70) priority=P3 sentiment=neu -> suggest conf=0.81
   draft: Thanks for getting in touch. Based on similar cases (track_order), the usual resolution is: Tracking link shar ...
   trace: fd4a022ecad34d739a6331a328a21d82 | tokens: 0 0 | cost: 0.0
t-2: intent=complaint (0.70) priority=P1 sentiment=neg -> escalate conf=0.49
   draft: Thanks for getting in touch. Based on similar cases (get_refund), the usual resolution is: Refund processed wi ...
   trace: 1b85c3ddf33149cca764cdd7d6d76d6b | tokens: 0 0 | cost: 0.0
t-3: intent=check_refund_policy (0.70) priority=P3 sentiment=neu -> auto_resolve conf=0.85
   draft: Thanks for getting in touch. Based on similar cases (check_refund_policy), the usual resolution is: 30-day ret ...
   trace: c7395aee4b76430b9edc7025c3bc4aa9 | tokens: 0 0 | cost: 0.0


## Audit log

Every triage writes an `AuditRecord`; the API exposes them at `GET /api/audit/recent` for the Kanban.

In [3]:
for row in asyncio.run(pipeline.audit.recent(5)):
    print(row.ticket_id, row.decision, f"{row.confidence:.2f}", row.llm_used,
          row.latency_ms, "ms")

t-3 auto_resolve 0.85 False 3 ms
t-2 escalate 0.49 False 3 ms
t-1 suggest 0.81 False 3 ms
